# **IMPORT LIBRARIES**

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
!pip install xgboost
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
pip install seaborn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:

import seaborn as sns

In [23]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

# **LOAD DATASET**

In [24]:

file_path = "../Data/eta_feature_engineered_dataset.csv"

In [25]:
import os
print("Current directory:", os.getcwd())
print("\nFiles in current directory:")
print(os.listdir())


Current directory: c:\Users\Asus\Documents\MDS\EY intership\ETA-Delay-Prediction-Logistics\notebooks

Files in current directory:
['01_EDA.ipynb', '02_Feature_engineering.ipynb', '03_Baseline_model.ipynb', '04_Feature_importance_selection.ipynb', '05_Classifiction_model.ipynb', '06_Regression_model.ipynb', '07_API_integration.ipynb', '08_Retrained_model.ipynb', 'ETA_Delay_Prediction_main.ipynb']


In [26]:

df = pd.read_csv(file_path)

In [27]:

print("Dataset Loaded Successfully")
print("Shape of dataset:", df.shape)

Dataset Loaded Successfully
Shape of dataset: (25000, 54)


# Regression Model

In [28]:
y = df['delay_hours_recon']

# Drop unnecessary columns
# FIX: Added speed_kmph_recon and speed_category — actual speed is only known
# AFTER delivery completes, so it cannot be used to predict delay hours.
df_model = df.drop(columns=[
    'delivery_id',
    'delay_hours_recon',
    'delivery_time_hours_recon',
    'delayed_flag_recon',
    'delayed',
    'delivery_status',
    'order_date_recon',
    'order_ts_recon',
    'delivery_ts_recon',
    'expected_ts_recon',
    'speed_kmph_recon',
    'speed_category',
])

X = df_model.copy()


In [29]:
# Numeric columns
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Categorical columns
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print("Numeric Features:", numeric_features)
print("Categorical Features:", categorical_features)

Numeric Features: ['distance_km', 'package_weight_kg', 'delivery_rating', 'delivery_cost', 'expected_time_hours_recon', 'weather_mult_recon', 'partner_mult_recon', 'hour', 'order_dayofweek', 'order_month', 'order_year', 'order_hour', 'is_weekend', 'rush_hour_flag', 'night_delivery_flag', 'delayed_flag', 'severe_delay_flag', 'efficiency_score', 'cost_per_km', 'cost_per_kg', 'heavy_flag', 'partner_delay_rate', 'region_delay_rate', 'vehicle_delay_rate', 'mode_delay_rate', 'bad_weather_flag', 'weather_severity', 'weather_distance_risk', 'status_delivered_flag', 'load_index', 'cost_weather_risk', 'partner_weather_risk', 'time_ratio', 'early_flag']
Categorical Features: ['delivery_partner', 'package_type', 'vehicle_type', 'delivery_mode', 'region', 'weather_condition', 'order_day_name', 'delay_severity']


In [30]:
# Preprocessing pipelines

numeric_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# RandomForestRegressor

In [32]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

In [34]:
rf_param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [None, 10, 20, 30],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4]
}

rf_search = RandomizedSearchCV(
    rf_pipeline,
    rf_param_grid,
    n_iter=15,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    random_state=42
)

rf_search.fit(X_train, y_train)

,estimator,Pipeline(step...m_state=42))])
,param_distributions,"{'model__max_depth': [None, 10, ...], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 5, ...], 'model__n_estimators': [100, 200, ...]}"
,n_iter,15
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [ ]:
rf_best = rf_search.best_estimator_

y_pred_rf = rf_best.predict(X_test)

print("Random Forest Results:")
print("MAE:", mean_absolute_error(y_test, y_pred_rf))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rf)))
print("R2:", r2_score(y_test, y_pred_rf))

Random Forest Results:
MAE: 0.0034123871404798094
RMSE: 0.023776709640286338
R2: 0.9999981876192516


# XGBoostRegressor

In [ ]:
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(
        objective='reg:squarederror',
        random_state=42
    ))
])

In [ ]:
xgb_param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [3, 5, 7],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__subsample': [0.8, 1.0],
    'model__colsample_bytree': [0.8, 1.0]
}

xgb_search = RandomizedSearchCV(
    xgb_pipeline,
    xgb_param_grid,
    n_iter=15,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    random_state=42
)

xgb_search.fit(X_train, y_train)

,estimator,"Pipeline(step...=None, ...))])"
,param_distributions,"{'model__colsample_bytree': [0.8, 1.0], 'model__learning_rate': [0.01, 0.05, ...], 'model__max_depth': [3, 5, ...], 'model__n_estimators': [100, 200, ...], ...}"
,n_iter,15
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [ ]:
xgb_best = xgb_search.best_estimator_

y_pred_xgb = xgb_best.predict(X_test)

print("XGBoost Results:")
print("MAE:", mean_absolute_error(y_test, y_pred_xgb))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_xgb)))
print("R2:", r2_score(y_test, y_pred_xgb))

XGBoost Results:
MAE: 0.023359024447584455
RMSE: 0.03528830277125517
R2: 0.9999960078442743


In [ ]:
results = pd.DataFrame({
    'Model': ['Random Forest', 'XGBoost'],
    'R2 Score': [
        r2_score(y_test, y_pred_rf),
        r2_score(y_test, y_pred_xgb)
    ],
    'MAE': [
        mean_absolute_error(y_test, y_pred_rf),
        mean_absolute_error(y_test, y_pred_xgb)
    ],
    'RMSE': [
        np.sqrt(mean_squared_error(y_test, y_pred_rf)),
        np.sqrt(mean_squared_error(y_test, y_pred_xgb))
    ]
})

print(results)

           Model  R2 Score       MAE      RMSE
0  Random Forest  0.999998  0.003412  0.023777
1        XGBoost  0.999996  0.023359  0.035288
